In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Ready to start training")

Ready to start training


In [9]:
df = pd.read_csv("nvda_complete_multimodal.csv",
                 index_col=[0],
                 parse_dates=True,
                 dayfirst=True)

print("Loaded shape:", df.shape)
print("\nAll columns:", df.columns.tolist())
print("\nLast 5 rows (key columns):")
print(df.tail(5)[['Close', 'finbert_sentiment', 'SMA_10', 'SMA_30', 'RSI', 'next_close', 'direction', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Middle', 'BB_Upper', 'BB_Lower']])

Loaded shape: (970, 787)

All columns: ['key_0', 'Close', 'High', 'Low', 'Open', 'Volume', 'finbert_sentiment', 'vit_dim_0', 'vit_dim_1', 'vit_dim_2', 'vit_dim_3', 'vit_dim_4', 'vit_dim_5', 'vit_dim_6', 'vit_dim_7', 'vit_dim_8', 'vit_dim_9', 'vit_dim_10', 'vit_dim_11', 'vit_dim_12', 'vit_dim_13', 'vit_dim_14', 'vit_dim_15', 'vit_dim_16', 'vit_dim_17', 'vit_dim_18', 'vit_dim_19', 'vit_dim_20', 'vit_dim_21', 'vit_dim_22', 'vit_dim_23', 'vit_dim_24', 'vit_dim_25', 'vit_dim_26', 'vit_dim_27', 'vit_dim_28', 'vit_dim_29', 'vit_dim_30', 'vit_dim_31', 'vit_dim_32', 'vit_dim_33', 'vit_dim_34', 'vit_dim_35', 'vit_dim_36', 'vit_dim_37', 'vit_dim_38', 'vit_dim_39', 'vit_dim_40', 'vit_dim_41', 'vit_dim_42', 'vit_dim_43', 'vit_dim_44', 'vit_dim_45', 'vit_dim_46', 'vit_dim_47', 'vit_dim_48', 'vit_dim_49', 'vit_dim_50', 'vit_dim_51', 'vit_dim_52', 'vit_dim_53', 'vit_dim_54', 'vit_dim_55', 'vit_dim_56', 'vit_dim_57', 'vit_dim_58', 'vit_dim_59', 'vit_dim_60', 'vit_dim_61', 'vit_dim_62', 'vit_dim_63', 'v

In [10]:
# All features to be used for training (prices, sentiment, indicators, ViT)
features = ['Open', 'High', 'Low', 'Close', 'Volume',
            'finbert_sentiment', 'SMA_10', 'SMA_30', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Middle', 'BB_Upper', 'BB_Lower'] + \
           [f"vit_dim_{i}" for i in range(768)]

# Scale everything
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[features])

# Targets
y_reg = df['next_close'].values
y_dir = df['direction'].values

# Create sequences
lookback = 250
X, y_reg_seq, y_dir_seq = [], [], []

for i in range(lookback, len(scaled)):
    X.append(scaled[i-lookback:i])
    y_reg_seq.append(y_reg[i])
    y_dir_seq.append(y_dir[i])

X = np.array(X)
y_reg_seq = np.array(y_reg_seq)
y_dir_seq = np.array(y_dir_seq)

print("Sequences ready")
print("X shape:", X.shape) 

Sequences ready
X shape: (720, 250, 783)


Train/Test Split

In [11]:
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_reg_train, y_reg_test = y_reg_seq[:train_size], y_reg_seq[train_size:]
y_dir_train, y_dir_test = y_dir_seq[:train_size], y_dir_seq[train_size:]

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

Train: 576 samples | Test: 144 samples


Dataset and Model

In [12]:
class StockDataset(Dataset):
    def __init__(self, X, y_reg, y_dir):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).unsqueeze(1)
        self.y_dir = torch.tensor(y_dir, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_reg[idx], self.y_dir[idx]

train_ds = StockDataset(X_train, y_reg_train, y_dir_train)
test_ds = StockDataset(X_test, y_reg_test, y_dir_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout)
        self.fc_reg = nn.Linear(hidden_size, 1)
        self.fc_class = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = h_n[-1]
        reg = self.fc_reg(h)
        class_out = torch.sigmoid(self.fc_class(h))
        return reg, class_out

model = StockLSTM(input_size=len(features))
print("Model created with", len(features), "input features")

Model created with 783 input features


Training

In [13]:
criterion_reg = nn.MSELoss()
criterion_class = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)  # lower lr for many features

device = torch.device("cpu")  # since no CUDA
model.to(device)

epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y_reg, batch_y_class in tqdm(train_loader):
        batch_x = batch_x.to(device)
        batch_y_reg = batch_y_reg.to(device)
        batch_y_class = batch_y_class.to(device)
        
        optimizer.zero_grad()
        reg_out, class_out = model(batch_x)
        loss = criterion_reg(reg_out, batch_y_reg) + criterion_class(class_out, batch_y_class)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.6f}")

  0%|          | 0/18 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:03<00:00,  4.75it/s]


Epoch 1/100 | Loss: 8347.861477


100%|██████████| 18/18 [00:02<00:00,  6.35it/s]


Epoch 2/100 | Loss: 8066.520828


...

Epoch 98/100 | Loss: 2962.443825


100%|██████████| 18/18 [00:02<00:00,  8.02it/s]


Epoch 99/100 | Loss: 2939.181994


100%|██████████| 18/18 [00:02<00:00,  7.85it/s]

Epoch 100/100 | Loss: 2916.269087


Evaluation

In [14]:
model.eval()  # switch to evaluation mode
reg_preds = []
class_preds = []
y_reg_true = []

with torch.no_grad():
    for batch_x, batch_y_reg, _ in test_loader:       
        batch_x = batch_x.to(device)
        reg_out, class_out = model(batch_x)
        
        reg_preds.extend(reg_out.cpu().numpy().flatten())
        class_preds.extend((class_out.cpu().numpy().flatten() > 0.5).astype(int))
        y_reg_true.extend(batch_y_reg.cpu().numpy().flatten())

# Calculate metrics
mae = mean_absolute_error(y_reg_test, reg_preds)
rmse = np.sqrt(mean_squared_error(y_reg_test, reg_preds))
dir_acc = accuracy_score(y_dir_test, class_preds)


print("\n=== FULL MULTIMODAL RESULTS (100 epochs) ===")
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Directional Accuracy: {dir_acc:.2%}")
# Quick comparison to previous baseline
print("\nPrevious baseline (prices + sentiment only):")
print("Test MAE: 121.9180 | RMSE: 126.8612 | Directional Accuracy: 53.13%")

print("\nImprovement in directional accuracy:", f"{dir_acc - 0.5313:.2%}" if dir_acc > 0.5313 else f"{dir_acc - 0.5313:.2%}")


=== FULL MULTIMODAL RESULTS (100 epochs) ===
Test MAE: 125.6851
Test RMSE: 126.7636
Directional Accuracy: 54.17%

Previous baseline (prices + sentiment only):
Test MAE: 121.9180 | RMSE: 126.8612 | Directional Accuracy: 53.13%

Improvement in directional accuracy: 1.04%


Save the model